# llm-training-kernels: GPU measurements

Runs `scripts/reproduce.sh`, which writes every result to `benchmarks/results/`. Works on Colab and in RunPod's Jupyter.

1. Pick a GPU runtime. A free **T4** runs the fp32 parts. An **L4 / A100 / RTX 4090** (sm80+) also runs everything bf16.
2. Run cells 1–4. Cell 4 covers tests, attention, model, profile and throughput, and needs no dataset.
3. Optional, cell 5: the full TinyStories pipeline (download, BPE, encode, 22.7M training). It takes hours, so use a paid or rented GPU.
4. Cell 6 zips `benchmarks/results/`. Send `gpu_results.zip` back.


In [ ]:
# 1. GPU + versions
!nvidia-smi
import platform

import torch
import triton

assert torch.cuda.is_available(), "No GPU: pick a GPU runtime"
cap = torch.cuda.get_device_capability()
print(
    "torch",
    torch.__version__,
    "| CUDA",
    torch.version.cuda,
    "| triton",
    triton.__version__,
    "| python",
    platform.python_version(),
)
print("GPU:", torch.cuda.get_device_name(), "| compute capability:", cap)
print("bf16 parts will", "run" if cap >= (8, 0) else "be SKIPPED (needs sm80+)")

In [ ]:
# 2. Get the code
import os

os.chdir("/content" if os.path.isdir("/content") else os.path.expanduser("~"))
!rm -rf llm-training-kernels && git clone -q https://github.com/GodD001/llm-training-kernels
os.chdir("llm-training-kernels")
!git log --oneline -1

In [ ]:
# 3. Install (keeps the preinstalled torch/triton)
!pip install -q -e '.[dev]'

In [ ]:
# 4. GPU stages that need no dataset: tests, attention, model, profile, throughput
!bash scripts/reproduce.sh

In [ ]:
# 5. OPTIONAL: full pipeline (TinyStories download -> BPE -> encode -> 22.7M training). Takes hours.
# !bash scripts/reproduce.sh data tokenizer encode train


In [ ]:
# 6. Zip results (and download on Colab)
!cd benchmarks && rm -f ../gpu_results.zip && zip -qr ../gpu_results.zip results -x 'results/*.nsys-rep'
try:
    from google.colab import files

    files.download("gpu_results.zip")
except ImportError:
    print("results in", os.path.abspath("gpu_results.zip"))